In [1]:
customers_df = spark.read.table("bronze_customers")
products_df = spark.read.table("bronze_products")
stores_df = spark.read.table("bronze_stores")
sales_df = spark.read.table("bronze_sales")
inventory_df = spark.read.table("bronze_inventory")

StatementMeta(, ea857d1a-20ba-445e-a850-f2a9d01eb43b, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.functions import col

print("CUSTOMERS")
customers_df.printSchema()
display(customers_df.limit(5))

print("PRODUCTS")
products_df.printSchema()
display(products_df.limit(5))

print("SALES")
sales_df.printSchema()
display(sales_df.limit(5))

StatementMeta(, ea857d1a-20ba-445e-a850-f2a9d01eb43b, 4, Finished, Available, Finished, False)

CUSTOMERS
root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- SignupDate: date (nullable = true)
 |-- CustomerSegment: string (nullable = true)



SynapseWidget(Synapse.DataFrame, d80c421d-1e64-44f5-a87d-e721edcf2853)

PRODUCTS
root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- UnitCost: double (nullable = true)
 |-- Supplier: string (nullable = true)
 |-- ActiveFlag: string (nullable = true)



SynapseWidget(Synapse.DataFrame, b075ac49-f7e9-431c-a14a-dd91ac692899)

SALES
root
 |-- SalesID: integer (nullable = true)
 |-- OrderDateTime: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- StoreID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- DiscountPct: integer (nullable = true)
 |-- SalesChannel: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- OrderStatus: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 328dba47-47ef-46e6-9818-123007b02247)

In [3]:
from pyspark.sql.functions import col, when, trim

# Check customer data quality issues
display(
    customers_df.filter(
        col("Email").isNull() |
        (trim(col("Email")) == "") |
        col("State").isNull() |
        (trim(col("State")) == "")
    )
)


StatementMeta(, ea857d1a-20ba-445e-a850-f2a9d01eb43b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e8dcb3c9-af7b-4e2c-83a3-d8a015f08c0c)

In [4]:
customers_silver_df = (
    customers_df
    .dropDuplicates(["CustomerID"])
    .withColumn(
        "Email",
        when(
            col("Email").isNull() | (trim(col("Email")) == ""),
            "unknown@example.com"
        ).otherwise(col("Email"))
    )
    .withColumn(
        "State",
        when(
            col("State").isNull() | (trim(col("State")) == ""),
            "UNKNOWN"
        ).otherwise(col("State"))
    )
)

StatementMeta(, ea857d1a-20ba-445e-a850-f2a9d01eb43b, 6, Finished, Available, Finished, False)

In [5]:
print("Bronze customer rows:", customers_df.count())
print("Silver customer rows:", customers_silver_df.count())

display(
    customers_silver_df.filter(
        (col("Email") == "unknown@example.com") |
        (col("State") == "UNKNOWN")
    )
)

StatementMeta(, ea857d1a-20ba-445e-a850-f2a9d01eb43b, 7, Finished, Available, Finished, False)

Bronze customer rows: 1000
Silver customer rows: 1000


SynapseWidget(Synapse.DataFrame, 5a011316-2149-404c-baac-076284d7820a)

In [6]:
customers_silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_customers")

StatementMeta(, ea857d1a-20ba-445e-a850-f2a9d01eb43b, 8, Finished, Available, Finished, False)

In [1]:
display(
    products_df.filter(
        col("Brand").isNull() |
        (trim(col("Brand")) == "") |
        (col("UnitPrice") <= 0)
    )
)

StatementMeta(, 2a924595-9f6a-40ab-9353-4df771d2da52, 3, Finished, Available, Finished, False)

NameError: name 'products_df' is not defined

In [2]:
from pyspark.sql.functions import col, when, trim

customers_df = spark.read.table("bronze_customers")
products_df = spark.read.table("bronze_products")
stores_df = spark.read.table("bronze_stores")
sales_df = spark.read.table("bronze_sales")
inventory_df = spark.read.table("bronze_inventory")

StatementMeta(, 2a924595-9f6a-40ab-9353-4df771d2da52, 4, Finished, Available, Finished, False)

In [3]:
print("Products rows:", products_df.count())
display(products_df.limit(5))

StatementMeta(, 2a924595-9f6a-40ab-9353-4df771d2da52, 5, Finished, Available, Finished, False)

Products rows: 200


SynapseWidget(Synapse.DataFrame, 64384a9a-4c41-486f-8d25-5dc2390cff9a)

In [4]:
display(
    products_df.filter(
        col("Brand").isNull() |
        (trim(col("Brand")) == "") |
        (col("UnitPrice") <= 0)
    )
)

StatementMeta(, 2a924595-9f6a-40ab-9353-4df771d2da52, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1811f3c0-5292-42ad-bf04-ed2f645a6c5b)

In [5]:
from pyspark.sql.functions import col, when, trim

products_silver_df = (
    products_df
    .dropDuplicates(["ProductID"])
    .withColumn(
        "Brand",
        when(
            col("Brand").isNull() | (trim(col("Brand")) == ""),
            "UNKNOWN"
        ).otherwise(col("Brand"))
    )
    .filter(col("UnitPrice") > 0)
)

StatementMeta(, 2a924595-9f6a-40ab-9353-4df771d2da52, 7, Finished, Available, Finished, False)

In [6]:
print("Bronze product rows:", products_df.count())
print("Silver product rows:", products_silver_df.count())

display(
    products_silver_df.filter(
        col("Brand") == "UNKNOWN"
    )
)

StatementMeta(, 2a924595-9f6a-40ab-9353-4df771d2da52, 8, Finished, Available, Finished, False)

Bronze product rows: 200
Silver product rows: 199


SynapseWidget(Synapse.DataFrame, 0d0b932e-b6f8-416c-812c-86c8c1661940)

In [7]:
products_silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_products")

StatementMeta(, 2a924595-9f6a-40ab-9353-4df771d2da52, 9, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.functions import col, trim, to_timestamp

display(
    sales_df.filter(
        (col("Quantity") <= 0) |
        col("CustomerID").isNull() |
        (trim(col("CustomerID")) == "") |
        col("TotalAmount").isNull() |
        (trim(col("TotalAmount")) == "")
    )
)

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 5, Finished, Available, Finished, False)

NameError: name 'sales_df' is not defined

In [4]:
from pyspark.sql.functions import col, trim, to_timestamp

customers_df = spark.read.table("bronze_customers")
products_df = spark.read.table("bronze_products")
stores_df = spark.read.table("bronze_stores")
sales_df = spark.read.table("bronze_sales")
inventory_df = spark.read.table("bronze_inventory")

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 6, Finished, Available, Finished, False)

In [5]:
print("Sales rows:", sales_df.count())
display(sales_df.limit(5))

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 7, Finished, Available, Finished, False)

Sales rows: 20000


SynapseWidget(Synapse.DataFrame, 6ebc9ae3-8ae2-4ad8-99f8-b7ae070def21)

In [6]:
display(
    sales_df.filter(
        (col("Quantity") <= 0) |
        col("CustomerID").isNull() |
        (trim(col("CustomerID")) == "") |
        col("TotalAmount").isNull() |
        (trim(col("TotalAmount")) == "")
    )
)

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, db0390f9-3441-448f-b803-812ad82be178)

In [7]:
from pyspark.sql.functions import to_timestamp, col

sales_date_check_df = sales_df.withColumn(
    "ParsedOrderDateTime",
    to_timestamp(col("OrderDateTime"), "yyyy-MM-dd HH:mm:ss")
)

display(
    sales_date_check_df.filter(
        col("ParsedOrderDateTime").isNull()
    )
)

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c07d6985-0650-4323-bc82-9cc0c8063425)

In [8]:
from pyspark.sql.functions import col, to_timestamp, when, round

sales_silver_df = (
    sales_df
    .withColumn(
        "OrderDateTime",
        to_timestamp(col("OrderDateTime"), "yyyy-MM-dd HH:mm:ss")
    )
    .filter(col("Quantity") > 0)
    .filter(col("CustomerID").isNotNull())
    .filter(col("OrderDateTime").isNotNull())
    .withColumn(
        "TotalAmount",
        when(
            col("TotalAmount").isNull(),
            round(
                col("Quantity") * col("UnitPrice") *
                (1 - col("DiscountPct") / 100),
                2
            )
        ).otherwise(col("TotalAmount"))
    )
)

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 10, Finished, Available, Finished, False)

In [9]:
sales_silver_df = sales_silver_df.dropDuplicates([
    "OrderDateTime",
    "CustomerID",
    "ProductID",
    "StoreID",
    "Quantity",
    "UnitPrice",
    "DiscountPct",
    "SalesChannel",
    "PaymentMethod"
])

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 11, Finished, Available, Finished, False)

In [10]:
print("Bronze sales rows:", sales_df.count())
print("Silver sales rows:", sales_silver_df.count())

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 12, Finished, Available, Finished, False)

Bronze sales rows: 20000
Silver sales rows: 19994


In [11]:
display(
    sales_silver_df.filter(
        (col("Quantity") <= 0) |
        col("CustomerID").isNull() |
        col("OrderDateTime").isNull() |
        col("TotalAmount").isNull()
    )
)

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fea5006b-f431-4cfc-a268-c115ae4b75d4)

In [12]:
sales_silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_sales")

StatementMeta(, dbe6c26b-29db-4e05-91fe-d51b0d22cb88, 14, Finished, Available, Finished, False)

In [1]:
from pyspark.sql.functions import col, trim

display(
    inventory_df.filter(
        (col("QuantityOnHand") < 0) |
        col("LastRestockDate").isNull() |
        (trim(col("LastRestockDate")) == "")
    )
)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 3, Finished, Available, Finished, False)

NameError: name 'inventory_df' is not defined

In [2]:
inventory_df = spark.read.table("bronze_inventory")

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 4, Finished, Available, Finished, False)

In [3]:
print("Inventory rows:", inventory_df.count())
display(inventory_df.limit(5))

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 5, Finished, Available, Finished, False)

Inventory rows: 4000


SynapseWidget(Synapse.DataFrame, 12f02193-1565-4579-8711-afeaa7b37704)

In [4]:
from pyspark.sql.functions import col, trim

display(
    inventory_df.filter(
        (col("QuantityOnHand") < 0) |
        col("LastRestockDate").isNull() |
        (trim(col("LastRestockDate")) == "")
    )
)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 503c2ffa-83a8-4b01-9cbc-5c93837648d0)

In [5]:
display(
    inventory_df.filter(
        (col("QuantityOnHand") < 0) |
        col("LastRestockDate").isNull()
    )
)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a7c5d76b-0f37-46f4-a926-a3dbe8be4f2f)

In [6]:
from pyspark.sql.functions import col, when, lit

inventory_silver_df = (
    inventory_df
    .dropDuplicates(["InventoryID"])
    
    # Remove impossible inventory quantities
    .filter(col("QuantityOnHand") >= 0)

    # Flag records with missing restock date
    .withColumn(
        "MissingRestockDateFlag",
        when(
            col("LastRestockDate").isNull(),
            lit("Y")
        ).otherwise(lit("N"))
    )
)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 8, Finished, Available, Finished, False)

In [8]:
print("Bronze inventory rows:", inventory_df.count())
print("Silver inventory rows:", inventory_silver_df.count())

display(
    inventory_silver_df.filter(
        col("MissingRestockDateFlag") == "Y"
    )
)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 10, Finished, Available, Finished, False)

Bronze inventory rows: 4000
Silver inventory rows: 3999


SynapseWidget(Synapse.DataFrame, 43647cc6-984f-403e-86e5-157eed8ded83)

In [9]:
inventory_silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_inventory")

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 11, Finished, Available, Finished, False)

In [10]:
display(stores_df)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 12, Finished, Available, Finished, False)

NameError: name 'stores_df' is not defined

In [11]:
stores_df = spark.read.table("bronze_inventory")

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 13, Finished, Available, Finished, False)

In [12]:
stores_df = spark.read.table("bronze_stores")

display(stores_df)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b13e25ed-b9f4-4063-8880-db386d721fb8)

In [13]:
from pyspark.sql.functions import col, trim

display(
    stores_df.filter(
        col("StoreID").isNull() |
        col("StoreName").isNull() |
        (trim(col("StoreName")) == "") |
        col("City").isNull() |
        (trim(col("City")) == "")
    )
)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 763c244a-7e9b-4d30-80ac-04ea5b86d6e0)

In [14]:
stores_silver_df = (
    stores_df
    .dropDuplicates(["StoreID"])
)

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 16, Finished, Available, Finished, False)

In [15]:
stores_silver_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_stores")

StatementMeta(, 5081a265-7a9d-437e-9dea-cf81ae15980d, 17, Finished, Available, Finished, False)